In [1]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# ED Ops Pipeline (Kaggle/Papermill) — Sim → Train (Focal) → Cal → Threshold (PR/OPS) → Replay (Top-K) → KPIs → Scenarios → Stress/Spot-Checks\n",
        "\n",
        "**Kaggle-ready**: kernelspec metadata, non-interactive plotting, env-aware paths, no widgets.\n",
        "Artifacts → **`/kaggle/working/artifacts/`** (or `./artifacts`).\n",
        "\n",
        "**What’s new in v2b**\n",
        "- 3 replays with explicit gate/policy reasons; **ownership** (BED after CONSULT), duplicate & stale capacity blocks.\n",
        "- Minutes-saved knobs: `MAX_CAP_AGE_MIN` (default 30) and `CAPACITY_DECAY ∈ {linear, convex2}`.\n",
        "- Pager burden: pages/hour histogram, optional **hard cap per 2-hour window**, hour-aware budgets hook saved to notes.\n",
        "- Safety: per-action precision/recall @ τ (OPS), τ sanity, misroutes table.\n",
        "- **Rule spot-check**: verifies that hand rules trigger the model’s action ≥80% of the time.\n",
        "- **Tau stress test**: lift critical recall floors to 0.90 + halve FP/hour budgets and compare pager load.\n",
        "- **Scenario runner**: 6 canonical tabletop cases with Top-K, gate reasons, minutes_saved.\n",
        "\n",
        "> Privacy/bias note: **No age/sex fields** are used in data or features."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Environment & reproducibility (Papermill-safe) ----",
        "import os, sys, json, math, random, numpy as np, pandas as pd",
        "import matplotlib",
        "matplotlib.use(\"Agg\")  # non-interactive backend",
        "import matplotlib.pyplot as plt",
        "import warnings",
        "from pathlib import Path",
        "from collections import Counter, defaultdict",
        "",
        "import torch, torch.nn as nn, torch.nn.functional as F",
        "",
        "SEED = 4242",
        "random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)",
        "",
        "ON_KAGGLE = Path('/kaggle/working').exists()",
        "WORKDIR   = Path('/kaggle/working') if ON_KAGGLE else Path('.')",
        "ARTS      = WORKDIR/'artifacts'; ARTS.mkdir(parents=True, exist_ok=True)",
        "",
        "# Data roots (recursive search under /kaggle/input)",
        "DATA_ROOTS = [Path('/kaggle/input'), Path('/mnt/data'), WORKDIR]",
        "def find_data_file(name):",
        "    for root in DATA_ROOTS:",
        "        if not root.exists():",
        "            continue",
        "        p = root/name",
        "        if p.exists():",
        "            return p",
        "        match = next((q for q in root.rglob(name) if q.is_file()), None)",
        "        if match is not None:",
        "            return match",
        "    return None",
        "",
        "print({\"on_kaggle\": ON_KAGGLE, \"workdir\": str(WORKDIR), \"arts\": str(ARTS)})",
        "warnings.filterwarnings(\"ignore\", category=UserWarning, module=\"sklearn.metrics._ranking\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Config knobs ----",
        "ACTIONS = ['NO_OP','ORDER_ECG','PERFORM_FAST','ORDER_LABS','ORDER_XR','ORDER_CT','REQUEST_CONSULT','REQUEST_BED']",
        "FEATURE_NAMES = ['minute_of_day','cap_stale','ems','consult_delay_min',",
        "                 'syn_chest_pain','syn_polytrauma','syn_neuro_deficit','syn_other',",
        "                 'ecg_hint','fast_hint','ct_hint','ems_prealert','risk_score']",
        "",
        "LAMBDA_BY_HOUR = {0:1.0,1:1.0,2:0.9,3:0.9,4:0.9,5:1.1,6:1.6,7:2.2,8:3.3,9:3.6,10:3.3,11:3.1,",
        "                  12:3.0,13:3.0,14:3.0,15:3.2,16:3.5,17:3.7,18:3.0,19:2.5,20:2.0,21:1.8,22:1.4,23:1.2}",
        "_LSUM = sum(LAMBDA_BY_HOUR.values())",
        "",
        "PRESET = \"ops_realistic\"            # or \"train_heavy\"",
        "TARGETS = {\"ops_realistic\": 100, \"train_heavy\": 300}",
        "TARGET_DAILY_ARRIVALS = TARGETS.get(PRESET, 100)",
        "LAMBDA_SCALE = TARGET_DAILY_ARRIVALS / _LSUM",
        "",
        "N_DAYS = 3",
        "SIM_MINUTES = N_DAYS * 24 * 60",
        "",
        "# Ops constraints",
        "RECALL_FLOOR = {'ORDER_ECG':0.85,'ORDER_CT':0.85,'PERFORM_FAST':0.85,'ORDER_LABS':0.60,'ORDER_XR':0.65}",
        "FP_BUDGET_PER_H = {\"REQUEST_CONSULT\": 0.10, \"REQUEST_BED\": 0.05}  # paging only",
        "",
        "# Hour-aware budgets hook (None means use FP_BUDGET_PER_H); example: {'REQUEST_CONSULT': {8:0.08, 9:0.08, ...}}",
        "HOUR_FP_BUDGETS = None",
        "",
        "# Optional hard cap for paging per 2-hour rolling window (None to disable)",
        "HARD_CAP_PAGES_PER_2H = 1",
        "",
        "# Top-K + mutual exclusions",
        "TOP_K = 2",
        "MUTEX = {frozenset({'ORDER_CT','PERFORM_FAST'})}",
        "",
        "# Focal loss",
        "USE_FOCAL = True",
        "GAMMA = 2.0",
        "",
        "# Minutes-saved knobs",
        "MAX_CAP_AGE_MIN = 30          # default 30; was 20",
        "CAPACITY_DECAY = \"linear\"     # 'linear' or 'convex2' (convex square)",
        "",
        "print(f\"Preset={PRESET} | target/day={TARGET_DAILY_ARRIVALS} | scale={LAMBDA_SCALE:.3f} | sim_minutes={SIM_MINUTES}\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Simulation or CSV loader (Papermill-safe) ----",
        "PREV = {'syn_chest_pain':0.22,'syn_polytrauma':0.06,'syn_neuro_deficit':0.08,'syn_other':0.64}",
        "",
        "def nonhom_poisson_arrivals(total_minutes, lam_by_hour, scale=1.0):",
        "    out=[]",
        "    for m in range(total_minutes):",
        "        h=(m//60)%24",
        "        lam = lam_by_hour.get(h,1.0)*scale/60.0",
        "        if np.random.rand()<lam: out.append(m)",
        "    return out",
        "",
        "def one_hot_syndrome():",
        "    r=np.random.rand()",
        "    if r<PREV['syn_chest_pain']: return 1,0,0,0",
        "    r-=PREV['syn_chest_pain']",
        "    if r<PREV['syn_polytrauma']: return 0,1,0,0",
        "    r-=PREV['syn_polytrauma']",
        "    if r<PREV['syn_neuro_deficit']: return 0,0,1,0",
        "    return 0,0,0,1",
        "",
        "def gen_dataset(total_minutes=SIM_MINUTES, scale=LAMBDA_SCALE, seq_len=5):",
        "    arr=nonhom_poisson_arrivals(total_minutes, LAMBDA_BY_HOUR, scale)",
        "    X=[]; y=[]; ts=[]",
        "    for t in arr:",
        "        ems = int(np.random.rand()<0.55)",
        "        ems_prealert = int(ems and (np.random.rand()<0.35))",
        "        cap_stale = int(np.random.rand()<0.25)",
        "        s_cp, s_poly, s_neuro, s_other = one_hot_syndrome()",
        "        ecg_hint = int(s_cp or (np.random.rand()<0.15))",
        "        fast_hint = int(s_poly or (np.random.rand()<0.10))",
        "        ct_hint = int((s_poly or s_neuro) or (np.random.rand()<0.10))",
        "        base_delay = np.random.normal(20, 8)",
        "        consult_delay_min_raw = max(0.0, base_delay - 6*ems_prealert + 5*cap_stale)",
        "        risk_raw = np.clip(0.15 + 0.35*s_poly + 0.35*s_neuro + 0.10*ems + 0.10*ems_prealert, 0, 1)",
        "        minute_of_day_raw = t % 1440",
        "",
        "        minute_of_day = minute_of_day_raw/1440.0",
        "        consult_delay_min = np.log1p(consult_delay_min_raw/10.0)",
        "        risk = risk_raw",
        "",
        "        feats = [minute_of_day, cap_stale, ems, consult_delay_min, s_cp, s_poly, s_neuro,",
        "                 s_other, ecg_hint, fast_hint, ct_hint, ems_prealert, risk]",
        "",
        "        # Base label (mirrors simple clinical rules)",
        "        if (ct_hint==1) and (s_neuro==1 or s_poly==1):",
        "            base = np.random.choice(['ORDER_CT','PERFORM_FAST'], p=[0.85,0.15])",
        "        elif (s_cp==1 and ecg_hint==1):",
        "            base = 'ORDER_ECG'",
        "        elif (risk>0.55 and ems==1):",
        "            base = 'ORDER_LABS'",
        "        elif (s_other==1 and fast_hint==0 and np.random.rand()<0.25):",
        "            base = 'ORDER_XR'",
        "        else:",
        "            base = 'NO_OP'",
        "",
        "        # Escalation to paging",
        "        escalate = (base in ['ORDER_CT','PERFORM_FAST','ORDER_LABS']) and (risk>0.65) and (np.random.rand()<0.35)",
        "        if escalate:",
        "            base = np.random.choice(['REQUEST_CONSULT','REQUEST_BED'], p=[0.75,0.25])",
        "",
        "        X.append(feats); y.append(ACTIONS.index(base)); ts.append(t)",
        "",
        "    X=np.array(X, dtype=np.float32); y=np.array(y, dtype=np.int64); ts=np.array(ts, dtype=np.int64)",
        "    X_seq = np.stack([X + 0.01*np.random.randn(*X.shape) for _ in range(seq_len)], axis=1).astype(np.float32)",
        "    return X_seq, y, ts",
        "",
        "csv_train = find_data_file('train_DE_full.csv')",
        "csv_val   = find_data_file('val_DE_full.csv')",
        "csv_test  = find_data_file('test_DE_full.csv')",
        "",
        "def maybe_autodetect_feature_names(df):",
        "    global FEATURE_NAMES",
        "    if not set(FEATURE_NAMES).issubset(df.columns):",
        "        cand = [c for c in df.columns if c not in {'label','ts'} and pd.api.types.is_numeric_dtype(df[c])]",
        "        FEATURE_NAMES = cand",
        "        print(\"[Auto] FEATURE_NAMES ->\", FEATURE_NAMES)",
        "",
        "if csv_train and csv_val and csv_test:",
        "    print(\"Loading provided CSVs from:\", csv_train.parent)",
        "    def load_split(p):",
        "        df = pd.read_csv(p)",
        "        maybe_autodetect_feature_names(df)",
        "        X = df[FEATURE_NAMES].values.astype('float32')",
        "        y = df['label'].map({k:i for i,k in enumerate(ACTIONS)}).values.astype('int64')",
        "        ts = df.get('ts', pd.Series(np.arange(len(df)))).values.astype('int64')",
        "        X_seq = np.stack([X + 0.01*np.random.randn(*X.shape) for _ in range(5)], axis=1).astype('float32')",
        "        return X_seq, y, ts",
        "    Xtr_all, ytr_all, ttr_all = load_split(csv_train)",
        "    Xcal_all, ycal_all, tcal_all = load_split(csv_val)",
        "    Xv_all,  yv_all,  tv_all  = load_split(csv_test)",
        "else:",
        "    print(\"Simulating dataset...\")",
        "    X_all, y_all, ts_all = gen_dataset()",
        "    print(\"Realized arrivals:\", len(X_all), f\"(~{len(X_all)/N_DAYS:.1f}/day)\")",
        "    idx = np.arange(len(y_all)); np.random.shuffle(idx)",
        "    n_cal = max(1, int(0.20*len(idx)))",
        "    n_val = max(1, int(0.20*len(idx)))",
        "    cal_idx = idx[:n_cal]; val_idx = idx[n_cal:n_cal+n_val]; train_idx = idx[n_cal+n_val:]",
        "    Xtr_all, ytr_all, ttr_all = X_all[train_idx], y_all[train_idx], ts_all[train_idx]",
        "    Xcal_all, ycal_all, tcal_all = X_all[cal_idx], y_all[cal_idx], ts_all[cal_idx]",
        "    Xv_all,  yv_all,  tv_all  = X_all[val_idx],  y_all[val_idx],  ts_all[val_idx]",
        "",
        "print(\"Shapes  train/cal/val:\", Xtr_all.shape, Xcal_all.shape, Xv_all.shape)",
        "print(\"Class dist (train):\", Counter(ytr_all))",
        "print(\"Class dist (cal):  \", Counter(ycal_all))",
        "print(\"Class dist (val):  \", Counter(yv_all))"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Model + Focal training ----",
        "device = torch.device(\"cpu\")",
        "",
        "class GRUHead(nn.Module):",
        "    def __init__(self, in_f, hidden=128, n_actions=8, dropout=0.2):",
        "        super().__init__()",
        "        self.gru = nn.GRU(in_f, hidden, batch_first=True)",
        "        self.drop = nn.Dropout(dropout)",
        "        self.fc = nn.Linear(hidden, n_actions)",
        "    def forward(self, x):",
        "        out,_=self.gru(x); h=self.drop(out[:,-1,:]); return self.fc(h)",
        "",
        "def class_weights(y, n_classes):",
        "    cnt = np.bincount(y, minlength=n_classes).astype(float)",
        "    inv = 1.0/np.sqrt(cnt + 1e-6)",
        "    w = inv / inv.sum() * n_classes",
        "    return torch.tensor(w, dtype=torch.float32)",
        "",
        "class FocalLoss(nn.Module):",
        "    def __init__(self, gamma=2.0, weight=None):",
        "        super().__init__()",
        "        self.gamma = gamma",
        "        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none')",
        "    def forward(self, logits, targets):",
        "        ce = self.ce(logits, targets)",
        "        with torch.no_grad():",
        "            pt = torch.softmax(logits, dim=-1).gather(1, targets[:,None]).squeeze(1).clamp_min(1e-6)",
        "        return ((1-pt)**self.gamma * ce).mean()",
        "",
        "def to_t(x): return torch.tensor(x, dtype=torch.float32, device=device)",
        "def to_y(x): return torch.tensor(x, dtype=torch.long, device=device)",
        "",
        "model = GRUHead(len(FEATURE_NAMES), hidden=128, n_actions=len(ACTIONS), dropout=0.2).to(device)",
        "weights = class_weights(ytr_all, len(ACTIONS)).to(device)",
        "criterion = FocalLoss(gamma=GAMMA, weight=weights) if USE_FOCAL else nn.CrossEntropyLoss(weight=weights)",
        "opt = torch.optim.AdamW(model.parameters(), lr=5e-4)",
        "sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=35)",
        "",
        "BATCH=256; EPOCHS=35",
        "def batches(X,y,bs=BATCH):",
        "    N=len(y); idx=np.arange(N); np.random.shuffle(idx)",
        "    for i in range(0,N,bs):",
        "        b=idx[i:i+bs]; yield X[b], y[b]",
        "",
        "for ep in range(1,EPOCHS+1):",
        "    model.train(); tl=0.0; tn=0",
        "    for xb,yb in batches(Xtr_all, ytr_all, BATCH):",
        "        xb_t, yb_t = to_t(xb), to_y(yb)",
        "        opt.zero_grad(); loss = criterion(model(xb_t), yb_t); loss.backward(); opt.step()",
        "        tl += float(loss.detach())*len(yb); tn += len(yb)",
        "    model.eval()",
        "    with torch.no_grad():",
        "        vl = float(criterion(model(to_t(Xcal_all)), to_y(ycal_all)).detach())",
        "    sched.step()",
        "    if ep%5==0 or ep in [1,2,3]: print(f\"Epoch {ep:02d}: train={tl/tn:.4f} cal={vl:.4f}\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Calibration on CAL + thresholds (PR-cost & OPS) ----",
        "@torch.no_grad()",
        "def logits_on(splitX): return model(to_t(splitX)).cpu().numpy()",
        "",
        "def softmaxT(z, T=1.0, axis=1):",
        "    zT = z / max(T,1e-6)",
        "    zT -= zT.max(axis=axis, keepdims=True)",
        "    e = np.exp(zT)",
        "    return e / e.sum(axis=axis, keepdims=True)",
        "",
        "def nll(probs, y_true):",
        "    idx=np.arange(len(y_true)); p=probs[idx, y_true]; return -np.log(np.clip(p,1e-9,1)).mean()",
        "",
        "cal_logits = logits_on(Xcal_all)",
        "Ts = np.linspace(0.6,1.8,25)",
        "bestT, best = 1.0, 1e9",
        "for T in Ts:",
        "    p = softmaxT(cal_logits, T=T)",
        "    cur = nll(p, ycal_all)",
        "    if cur < best:",
        "        best, bestT = cur, T",
        "print(f\"[Calib] Best T on CAL = {bestT:.3f}\")",
        "",
        "probs_cal = softmaxT(cal_logits, T=bestT)",
        "",
        "def pick_tau_cost(y_true_bin, y_prob, fp_cost=1.0, fn_cost=5.0):",
        "    taus = np.linspace(0,1,101); best=None",
        "    for t in taus:",
        "        yp=(y_prob>=t).astype(int)",
        "        tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())",
        "        prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)",
        "        cost=fp_cost*fp + fn_cost*fn",
        "        if best is None or cost<best[0]: best=(cost,t,prec,rec,f1,tp,fp,fn)",
        "    c,t,prec,rec,f1,tp,fp,fn=best",
        "    return {\"tau\":float(t),\"precision\":float(prec),\"recall\":float(rec),\"f1\":float(f1),",
        "            \"cost\":float(c),\"tp\":tp,\"fp\":fp,\"fn\":fn}",
        "",
        "def pick_tau_ops(y_true_bin, y_prob, ts_min, name):",
        "    taus=np.linspace(0,1,101)",
        "    hours=(ts_min.max()-ts_min.min()+1)/60.0",
        "    best=None",
        "    recall_floor = RECALL_FLOOR.get(name, None)",
        "    fp_budget_per_h = FP_BUDGET_PER_H.get(name, None)",
        "    for t in taus:",
        "        yp=(y_prob>=t).astype(int)",
        "        tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())",
        "        prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)",
        "        fp_per_h = fp / max(hours,1e-6)",
        "        if recall_floor is not None and rec < recall_floor: continue",
        "        if fp_budget_per_h is not None and fp_per_h > fp_budget_per_h: continue",
        "        score=f1",
        "        if best is None or score>best[0]: best=(score,t,prec,rec,f1,tp,fp,fn,fp_per_h)",
        "    if best is None:",
        "        paging = name in ['REQUEST_CONSULT','REQUEST_BED']",
        "        if not paging:",
        "            best2=None",
        "            for t in np.linspace(0,1,101):",
        "                yp=(y_prob>=t).astype(int)",
        "                tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())",
        "                prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)",
        "                fp_per_h = fp / max(hours,1e-6)",
        "                score=f1",
        "                if best2 is None or score>best2[0]: best2=(score,t,prec,rec,f1,tp,fp,fn,fp_per_h)",
        "            score,t,prec,rec,f1,tp,fp,fn,fp_per_h = best2",
        "            return {\"tau\":float(t),\"precision\":float(prec),\"recall\":float(rec),\"f1\":float(f1),",
        "                    \"tp\":tp,\"fp\":fp,\"fn\":fn,\"fp_per_h\":float(fp_per_h),\"status\":\"relaxed\",\"relaxed_from\":recall_floor}",
        "        else:",
        "            t=0.5; yp=(y_prob>=t).astype(int)",
        "            tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())",
        "            prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)",
        "            fp_per_h = fp / max(hours,1e-6)",
        "            return {\"tau\":float(t),\"precision\":float(prec),\"recall\":float(rec),\"f1\":float(f1),",
        "                    \"tp\":tp,\"fp\":fp,\"fn\":fn,\"fp_per_h\":float(fp_per_h),\"status\":\"fallback\"}",
        "    score,t,prec,rec,f1,tp,fp,fn,fp_per_h = best",
        "    return {\"tau\":float(t),\"precision\":float(prec),\"recall\":float(rec),\"f1\":float(f1),",
        "            \"tp\":tp,\"fp\":fp,\"fn\":fn,\"fp_per_h\":float(fp_per_h)}",
        "",
        "thr_cost, thr_ops, notes = {}, {}, {}",
        "for ci,name in enumerate(ACTIONS):",
        "    yb=(ycal_all==ci).astype(int)",
        "    thr_cost[name] = pick_tau_cost(yb, probs_cal[:,ci])",
        "    thr_ops[name]  = pick_tau_ops(yb, probs_cal[:,ci], tcal_all, name)",
        "    notes[name] = {'recall_floor': RECALL_FLOOR.get(name),",
        "                   'fp_budget_per_h': FP_BUDGET_PER_H.get(name),",
        "                   'achieved_ops': thr_ops[name]}",
        "",
        "# Persist thresholds",
        "(ARTS/'thresholds.json').write_text(json.dumps(thr_cost, indent=2))",
        "(ARTS/'thresholds_ops.json').write_text(json.dumps(thr_ops, indent=2))",
        "",
        "# Add hour-aware budgets (if present) to notes for review",
        "if HOUR_FP_BUDGETS:",
        "    for a, hmap in HOUR_FP_BUDGETS.items():",
        "        notes.setdefault(a, {})['hour_fp_budget'] = hmap",
        "(ARTS/'thresholds_ops_notes.json').write_text(json.dumps(notes, indent=2))",
        "",
        "bad = [n for n,info in thr_ops.items() if info.get('status')=='fallback']",
        "if bad:",
        "    print(f\"[WARN] Ops τ fallback (paging strict): {bad}\")",
        "print(\"Saved thresholds & notes.\")",
        "",
        "print(\"\\nτ by action (PR-cost → OPS):\")",
        "for a in ACTIONS:",
        "    if a=='NO_OP': continue",
        "    tc = thr_cost.get(a,{}).get('tau', None)",
        "    to = thr_ops.get(a,{}).get('tau', None)",
        "    print(f\"  {a:18s}  {tc if tc is not None else '<none>'}  →  {to if to is not None else '<none>'}\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- VAL booster (≥15 positives for CONSULT/BED) + raise common floors to 0.80 ----",
        "MIN_POS = 15",
        "PAGING  = {'REQUEST_CONSULT','REQUEST_BED'}",
        "name_for = {i:n for i,n in enumerate(ACTIONS)}",
        "idx_for  = {n:i for i,n in enumerate(ACTIONS)}",
        "",
        "def boost_val_support(Xv, yv, tv, Xpools, min_pos=MIN_POS):",
        "    from collections import Counter",
        "    X_out, y_out, t_out = [Xv.copy()], [yv.copy()], [tv.copy()]",
        "    val_counts = Counter(yv)",
        "    for a in PAGING:",
        "        ai = idx_for[a]",
        "        have = int(val_counts.get(ai, 0))",
        "        need = max(0, min_pos - have)",
        "        if need <= 0: continue",
        "        Xpos = []",
        "        for Xp, yp in Xpools:",
        "            Xpos.append(Xp[yp==ai])",
        "        Xpos = np.concatenate([*Xpos, Xv[yv==ai]], axis=0) if Xpos else Xv[yv==ai]",
        "        if Xpos.size == 0:",
        "            print(f\"[VAL boost] No positives to clone for {a}\")",
        "            continue",
        "        take = Xpos[np.random.choice(len(Xpos), size=need, replace=True)]",
        "        jitter = 0.01*np.random.randn(*take.shape).astype(np.float32)",
        "        X_aug = (take + jitter).astype(np.float32)",
        "        y_aug = np.full((need,), ai, dtype=np.int64)",
        "        t_aug = np.arange(tv.max()+1, tv.max()+1+need, dtype=np.int64)",
        "        X_out.append(X_aug); y_out.append(y_aug); t_out.append(t_aug)",
        "        print(f\"[VAL boost] Added {need} synthetic {a} positives (target ≥ {min_pos})\")",
        "    return np.concatenate(X_out), np.concatenate(y_out), np.concatenate(t_out)",
        "",
        "Xv_all, yv_all, tv_all = boost_val_support(",
        "    Xv_all, yv_all, tv_all,",
        "    Xpools=[(Xtr_all, ytr_all), (Xcal_all, ycal_all)],",
        "    min_pos=MIN_POS",
        ")",
        "",
        "# Raise common action floors and regenerate OPS τ",
        "RECALL_FLOOR.update({'ORDER_ECG':0.80, 'ORDER_LABS':0.80, 'ORDER_XR':0.80})",
        "probs_cal = softmaxT(logits_on(Xcal_all), T=bestT)",
        "thr_ops = {}",
        "for ci,name in enumerate(ACTIONS):",
        "    yb=(ycal_all==ci).astype(int)",
        "    thr_ops[name]  = pick_tau_ops(yb, probs_cal[:,ci], tcal_all, name)",
        "(ARTS/'thresholds_ops.json').write_text(json.dumps(thr_ops, indent=2))",
        "print(\"[OPS] Regenerated τ with recall floors for common actions (ECG/LABS/XR=0.80)\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Replay on VAL (Top-K, gate vs policy, ownership, capacity, hour caps) ----",
        "@torch.no_grad()",
        "def predict_probs_row(x_row, T):",
        "    logits = model(to_t(x_row[None,...]))",
        "    return torch.softmax(logits/T, dim=-1).cpu().numpy()[0]",
        "",
        "def capacity_discount(ts_now, last_update_ts, max_age_min=MAX_CAP_AGE_MIN, mode=CAPACITY_DECAY):",
        "    if last_update_ts is None: return 0.0",
        "    delta = ts_now - last_update_ts",
        "    if delta <= 0: return 1.0",
        "    if delta >= max_age_min: return 0.0",
        "    x = delta / max_age_min",
        "    if mode == 'convex2':",
        "        return max(0.0, 1.0 - x*x)",
        "    return max(0.0, 1.0 - x)",
        "",
        "def minutes_saved_for(action, ts_now, last_cap_update):",
        "    hour = (ts_now//60)%24",
        "    is_day = 8 <= hour < 20",
        "    if action in ['ORDER_CT','PERFORM_FAST']:",
        "        base = 15.0 if is_day else 30.0",
        "    elif action in ['REQUEST_CONSULT','REQUEST_BED']:",
        "        base = 4.0 if is_day else 6.0",
        "    elif action in ['ORDER_ECG','ORDER_LABS']:",
        "        base = 2.0 if is_day else 3.0",
        "    elif action in ['ORDER_XR']:",
        "        base = 1.5 if is_day else 2.5",
        "    else:",
        "        base = 0.0",
        "    return base * capacity_discount(ts_now, last_cap_update)",
        "",
        "def replay(thresholds, Tcal, X_ref, y_ref, ts_ref, policy_on=True, tau_source=\"\"):",
        "    logs={'proposals':[], 'gate_blocks':[], 'policy_blocks':[], 'approvals':[]}",
        "    state={'last_capacity_update_ts':None, 'sent_packets':set(), 'consult_ok':set()}",
        "    # capacity updates schedule",
        "    t=0; caps=set()",
        "    while t<SIM_MINUTES: caps.add(t); t+=int(np.random.uniform(10,20))",
        "    # rolling paging cap (per 2h)",
        "    paging_times = []  # minutes for approved/proposed pages for cap checks",
        "    order=np.argsort(ts_ref)",
        "    for i in order:",
        "        tnow=int(ts_ref[i])",
        "        if tnow in caps: state['last_capacity_update_ts']=tnow",
        "        pr = predict_probs_row(X_ref[i], Tcal)",
        "        # tau gating",
        "        cands=[(ACTIONS[c], float(pr[c]), {'tau': thresholds.get(ACTIONS[c],{}).get('tau',0.5),",
        "                                           'tau_src': tau_source, 'p': float(pr[c])})",
        "               for c in range(len(ACTIONS)) if ACTIONS[c] != 'NO_OP']",
        "        below = [(n,p,m) for (n,p,m) in cands if p < m['tau']]",
        "        for n,p,m in below:",
        "            logs['gate_blocks'].append({'idx':int(i),'action':n,'prob':p,'tau':m['tau'],'reason':'below_tau'})",
        "        cands = [(n,p,m) for (n,p,m) in cands if p >= m['tau']]",
        "        # sort & mutex",
        "        cands.sort(key=lambda x:-x[1])",
        "        selected=[]",
        "        for n,p,meta in cands:",
        "            if len(selected) >= TOP_K: break",
        "            if any(frozenset({n,m}) in MUTEX for m,_,_ in selected):",
        "                continue",
        "            selected.append((n,p,meta))",
        "        if not selected: continue",
        "        # optional deterministic gates",
        "        packet_complete = True",
        "        trauma_cleared  = True",
        "        for name,score,meta in selected:",
        "            if policy_on and name in {'REQUEST_CONSULT','REQUEST_BED'}:",
        "                # capacity freshness gate",
        "                if state.get('last_capacity_update_ts') is None or (tnow - state['last_capacity_update_ts']) > MAX_CAP_AGE_MIN:",
        "                    logs['policy_blocks'].append({'idx':int(i),'action':name,'reason':'capacity_stale'})",
        "                    continue",
        "                # packet completeness + trauma ownership",
        "                if not packet_complete:",
        "                    logs['policy_blocks'].append({'idx':int(i),'action':name,'reason':'packet_incomplete'})",
        "                    continue",
        "                if name=='REQUEST_BED' and not trauma_cleared:",
        "                    logs['policy_blocks'].append({'idx':int(i),'action':name,'reason':'ownership'})",
        "                    continue",
        "                # duplicate gate",
        "                key=(int(i), name, 'IM')",
        "                if key in state['sent_packets']:",
        "                    logs['policy_blocks'].append({'idx':int(i),'action':name,'reason':'duplicate'})",
        "                    continue",
        "                # rolling cap per 2 hours",
        "                if HARD_CAP_PAGES_PER_2H is not None:",
        "                    window_start = tnow - 120",
        "                    paging_times = [t for t in paging_times if t >= window_start]",
        "                    if len(paging_times) >= HARD_CAP_PAGES_PER_2H:",
        "                        logs['policy_blocks'].append({'idx':int(i),'action':name,'reason':'cap_2h'})",
        "                        continue",
        "                state['sent_packets'].add(key)",
        "            # proposal",
        "            logs['proposals'].append({'idx':int(i),'action':name,'p':score,'ts':tnow})",
        "            # approvals with ownership effect",
        "            if name=='REQUEST_CONSULT':",
        "                if np.random.rand()<0.7:",
        "                    logs['approvals'].append({'idx':int(i),'action':name,'ts':tnow})",
        "                    state['consult_ok'].add(int(i))",
        "                    paging_times.append(tnow)",
        "            elif name=='REQUEST_BED':",
        "                if int(i) in state['consult_ok'] and np.random.rand()<0.7:",
        "                    logs['approvals'].append({'idx':int(i),'action':name,'ts':tnow})",
        "                    paging_times.append(tnow)",
        "            else:",
        "                logs['approvals'].append({'idx':int(i),'action':name,'ts':tnow})",
        "    return logs",
        "",
        "val_logits = logits_on(Xv_all)",
        "probs_val = softmaxT(val_logits, T=bestT)",
        "thr_cost = json.loads((ARTS/'thresholds.json').read_text())",
        "thr_ops  = json.loads((ARTS/'thresholds_ops.json').read_text())",
        "",
        "logs_cost_raw = replay(thr_cost, bestT, Xv_all, yv_all, tv_all, policy_on=False, tau_source=\"cost_raw_no_policy\")",
        "logs_cost     = replay(thr_cost, bestT, Xv_all, yv_all, tv_all, policy_on=True,  tau_source=\"cost_with_policy\")",
        "logs_ops      = replay(thr_ops,  bestT, Xv_all, yv_all, tv_all, policy_on=True,  tau_source=\"ops_with_policy\")",
        "",
        "from collections import Counter as C",
        "def summarize(logs, label):",
        "    props=len(logs['proposals']); appr=len(logs['approvals'])",
        "    print(f\"{label}: proposals={props} approvals={appr} approve_rate={appr/max(1,props):.2f} \"",
        "          f\"gate_blocks={len(logs['gate_blocks'])} policy_blocks={len(logs['policy_blocks'])}\")",
        "    if logs['gate_blocks']:",
        "        print(\"Gate blocks (reasons):\", C([g['reason'] for g in logs['gate_blocks']]))",
        "    if logs['policy_blocks']:",
        "        print(\"Policy blocks (reasons):\", C([p['reason'] for p in logs['policy_blocks']]))",
        "",
        "summarize(logs_cost_raw, \"Replay (PR/Cost τ, policy OFF, VAL)\")",
        "summarize(logs_cost,     \"Replay (PR/Cost τ, policy ON,  VAL)\")",
        "summarize(logs_ops,      \"Replay (OPS τ, policy ON, VAL)\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Per-action audits, support checks, pages/hour histogram, equity hour check ----",
        "def by_action_counts(logs):",
        "    props = Counter([p['action'] for p in logs['proposals']])",
        "    gblk  = Counter([g['action'] for g in logs['gate_blocks']])",
        "    pblk  = Counter([p['action'] for p in logs['policy_blocks']])",
        "    appr  = Counter([a['action'] for a in logs['approvals']])",
        "    acts  = [a for a in ACTIONS if a!='NO_OP']",
        "    rows=[]",
        "    for a in acts:",
        "        rows.append({",
        "            'action': a,",
        "            'proposals': int(props[a]),",
        "            'approvals': int(appr[a]),",
        "            'approve_rate': round(appr[a]/max(1,props[a]),3),",
        "            'gate_blocks': int(gblk[a]),",
        "            'policy_blocks': int(pblk[a]),",
        "        })",
        "    return rows",
        "",
        "def print_table(rows, title):",
        "    print(f\"\\n{title}\")",
        "    print(\"action\".ljust(18), \"props\".rjust(6), \"appr\".rjust(6), \"appr_rate\".rjust(10),",
        "          \"gate_blk\".rjust(9), \"policy_blk\".rjust(11))",
        "    for r in rows:",
        "        print(r['action'].ljust(18),",
        "              str(r['proposals']).rjust(6),",
        "              str(r['approvals']).rjust(6),",
        "              f\"{r['approve_rate']:.3f}\".rjust(10),",
        "              str(r['gate_blocks']).rjust(9),",
        "              str(r['policy_blocks']).rjust(11))",
        "",
        "print_table(by_action_counts(logs_cost_raw), \"Replay #1 — PR-cost τ, policy OFF (by action)\")",
        "print_table(by_action_counts(logs_cost),     \"Replay #2 — PR-cost τ, policy ON  (by action)\")",
        "print_table(by_action_counts(logs_ops),      \"Replay #3 — OPS τ, policy ON     (by action)\")",
        "",
        "# support integrity for VAL",
        "val_counts = Counter(yv_all)",
        "idx_for = {n:i for i,n in enumerate(ACTIONS)}",
        "print(f\"\\n[Support] VAL REQUEST_CONSULT >=15: {val_counts.get(idx_for['REQUEST_CONSULT'],0) >= 15} (\"",
        "      f\"{val_counts.get(idx_for['REQUEST_CONSULT'],0)})\")",
        "print(f\"[Support] VAL REQUEST_BED     >=15: {val_counts.get(idx_for['REQUEST_BED'],0) >= 15} (\"",
        "      f\"{val_counts.get(idx_for['REQUEST_BED'],0)})\")",
        "",
        "# pages/hour histogram (OPS proposals)",
        "ph = defaultdict(int)",
        "for p in logs_ops['proposals']:",
        "    if p['action'] in ['REQUEST_CONSULT','REQUEST_BED']:",
        "        h = (p['ts']//60)%24",
        "        ph[h]+=1",
        "print(\"\\nPages/hour (OPS proposals):\")",
        "for h in range(24):",
        "    print(f\"  {h:02d}: {ph[h]}\")",
        "",
        "# equity sanity: ensure hour windows aren't systematically under-paged",
        "day_pages   = sum(ph[h] for h in range(8,20))",
        "night_pages = sum(ph[h] for h in [0,1,2,3,4,5,6,7,20,21,22,23])",
        "print(f\"[Equity-hour] day_pages={day_pages}, night_pages={night_pages}. If night is budget-constrained, consider hour-aware FP budgets or caps.\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Precision/Recall @ chosen τ (OPS) + misroutes ----",
        "def per_action_pr_at_tau(probs, y_true, thr):",
        "    out=[]",
        "    for a in ACTIONS:",
        "        if a=='NO_OP': continue",
        "        ci = ACTIONS.index(a)",
        "        yb = (y_true==ci).astype(int)",
        "        tau = thr.get(a,{}).get('tau', 0.5)",
        "        yp = (probs[:,ci] >= tau).astype(int)",
        "        tp = int(((yp==1)&(yb==1)).sum()); fp=int(((yp==1)&(yb==0)).sum())",
        "        fn = int(((yp==0)&(yb==1)).sum())",
        "        prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)",
        "        out.append((a, round(prec,3), round(rec,3), tau, tp, fp, fn))",
        "    return out",
        "",
        "print(\"\\nPrecision/Recall at OPS τ (VAL):\")",
        "for a,prec,rec,tau,tp,fp,fn in per_action_pr_at_tau(probs_val, yv_all, thr_ops):",
        "    print(f\"  {a:18s} P={prec:.3f} R={rec:.3f} τ={tau:.2f}  (tp={tp}, fp={fp}, fn={fn})\")",
        "",
        "# Misroutes: true critical predicted as other imaging",
        "crit=set(['ORDER_CT','PERFORM_FAST','ORDER_ECG'])",
        "y_pred = probs_val.argmax(axis=1)",
        "misroutes = defaultdict(int)",
        "for t,p in zip(yv_all, y_pred):",
        "    true = ACTIONS[t]; pred = ACTIONS[p]",
        "    if true in crit and pred not in [true,'NO_OP']:",
        "        misroutes[(true,pred)] += 1",
        "if misroutes:",
        "    print(\"\\nMisroutes (true → pred, count):\")",
        "    for (t,p),c in misroutes.items():",
        "        print(f\"  {t} → {p}: {c}\")",
        "else:",
        "    print(\"\\nMisroutes: none observed\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Rule spot-check: do simple heuristics match model proposals ≥80%? ----",
        "def rule_action(row):",
        "    s_cp, s_poly, s_neuro = int(row[4]), int(row[5]), int(row[6])",
        "    ecg_hint, fast_hint, ct_hint = int(row[8]), int(row[9]), int(row[10])",
        "    risk = float(row[12]); ems = int(row[2])",
        "    if (ct_hint==1) and (s_neuro==1 or s_poly==1):",
        "        return 'ORDER_CT'",
        "    if (s_cp==1 and ecg_hint==1):",
        "        return 'ORDER_ECG'",
        "    if (risk>0.55 and ems==1):",
        "        return 'ORDER_LABS'",
        "    if (int(row[7])==1 and fast_hint==0):",
        "        return 'ORDER_XR'",
        "    return 'NO_OP'",
        "",
        "ops_tau = json.loads((ARTS/'thresholds_ops.json').read_text())",
        "agree = 0; total = 0",
        "for i in range(len(Xv_all)):",
        "    ra = rule_action(Xv_all[i,0,:])  # evaluate rules on current features",
        "    if ra == 'NO_OP': continue",
        "    ci = ACTIONS.index(ra)",
        "    p = probs_val[i, ci]",
        "    tau = ops_tau.get(ra, {}).get('tau', 0.5)",
        "    total += 1",
        "    agree += int(p >= tau)",
        "pct = (agree / max(1,total)) if total else 0.0",
        "print(f\"[Rule spot-check] {agree}/{total} ({pct:.1%}) of rule-fired cases would pass OPS τ. Target: ≥80%.\")"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Scenario runner (6 cards) ----",
        "def make_case(minute_of_day, cap_stale, ems, consult_delay_min, s_cp, s_poly, s_neuro, s_other,",
        "              ecg_hint, fast_hint, ct_hint, ems_prealert, risk):",
        "    x = np.array([[minute_of_day, cap_stale, ems, consult_delay_min, s_cp, s_poly, s_neuro, s_other,",
        "                   ecg_hint, fast_hint, ct_hint, ems_prealert, risk]], dtype=np.float32)",
        "    X_seq = np.stack([x + 0.01*np.random.randn(*x.shape) for _ in range(5)], axis=1).astype(np.float32)",
        "    return X_seq",
        "",
        "def explain_case(name, X_seq, ts_min, thresholds, T, top_k=TOP_K):",
        "    pr = predict_probs_row(X_seq[0], T)",
        "    pairs = [(ACTIONS[i], float(pr[i]), thresholds.get(ACTIONS[i],{}).get('tau',0.5)) for i in range(len(ACTIONS)) if ACTIONS[i] != 'NO_OP']",
        "    pairs.sort(key=lambda x:-x[1])",
        "    chosen=[]; reasons=[]",
        "    last_cap_update = ts_min if np.random.rand()<0.9 else ts_min-45  # simulate freshness 90%",
        "    for a,p,tau in pairs:",
        "        if len(chosen)>=top_k: break",
        "        if p < tau:",
        "            reasons.append((a,p,tau,'below_tau'))",
        "            continue",
        "        if any(frozenset({a,m}) in MUTEX for m,_ in chosen):",
        "            reasons.append((a,p,tau,'mutex'))",
        "            continue",
        "        chosen.append((a,p))",
        "    ms = {a: round(minutes_saved_for(a, ts_min, last_cap_update),2) for a,_ in chosen}",
        "    print(f\"\\n[Scenario] {name}\")",
        "    print(\"Top-K proposals:\")",
        "    for a,p in chosen:",
        "        print(f\"  {a:18s} p={p:.3f} τ={thresholds.get(a,{}).get('tau',0.5):.2f} minutes_saved≈{ms[a]}\")",
        "    if reasons:",
        "        print(\"Skipped / reasons:\")",
        "        for a,p,tau,r in reasons[:6]:",
        "            print(f\"  {a:18s} p={p:.3f} τ={tau:.2f} reason={r}\")",
        "",
        "def run_scenarios():",
        "    cards=[]",
        "    cards.append((\"STEMI day\", make_case(9/24, 0, 1, np.log1p(15/10), 1,0,0,0, 1,0,0,1, 0.6), 9*60))",
        "    cards.append((\"Neuro deficit night\", make_case(2/24, 0, 1, np.log1p(20/10), 0,0,1,0, 0,0,1,0, 0.7), 2*60))",
        "    cards.append((\"Polytrauma on AC\", make_case(15/24, 0, 1, np.log1p(10/10), 0,1,0,0, 0,1,1,1, 0.9), 15*60))",
        "    cards.append((\"Sepsis day\", make_case(10/24, 0, 1, np.log1p(25/10), 0,0,0,1, 0,0,0,1, 0.7), 10*60))",
        "    cards.append((\"Sepsis night\", make_case(23/24, 0, 1, np.log1p(25/10), 0,0,0,1, 0,0,0,1, 0.7), 23*60))",
        "    cards.append((\"Low-acuity crowding\", make_case(14/24, 1, 0, np.log1p(35/10), 0,0,0,1, 0,0,0,0, 0.2), 14*60))",
        "    cards.append((\"Borderline stale capacity\", make_case(18/24, 1, 1, np.log1p(18/10), 0,1,0,0, 0,1,1,1, 0.8), 18*60))",
        "    print(\"\\n=== Scenario Cards (OPS τ) ===\")",
        "    for name,X,ts in cards:",
        "        explain_case(name, X, ts, thr_ops, bestT, TOP_K)",
        "",
        "run_scenarios()"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Tau stress test: higher recall floors + tighter FP budgets ----",
        "def pager_per_hour(logs):",
        "    pages = [p for p in logs['proposals'] if p['action'] in ['REQUEST_CONSULT','REQUEST_BED']]",
        "    hours = max(1.0, (tv_all.max()-tv_all.min())/60.0)",
        "    return round(len(pages)/hours,3)",
        "",
        "stress_RECALL = {'ORDER_CT':0.90,'PERFORM_FAST':0.90,'ORDER_ECG':0.90}",
        "stress_FP = {k:v/2 for k,v in FP_BUDGET_PER_H.items()}",
        "",
        "orig_RF = RECALL_FLOOR.copy(); orig_FP = FP_BUDGET_PER_H.copy()",
        "RECALL_FLOOR.update(stress_RECALL); FP_BUDGET_PER_H.update(stress_FP)",
        "",
        "probs_cal = softmaxT(logits_on(Xcal_all), T=bestT)",
        "thr_ops_stress = {}",
        "for ci,name in enumerate(ACTIONS):",
        "    yb=(ycal_all==ci).astype(int)",
        "    thr_ops_stress[name]=pick_tau_ops(yb, probs_cal[:,ci], tcal_all, name)",
        "",
        "logs_ops_stress = replay(thr_ops_stress, bestT, Xv_all, yv_all, tv_all, policy_on=True, tau_source=\"ops_stress\")",
        "print(\"[Stress] OPS pages/hour:\", pager_per_hour(logs_ops), \"→\", pager_per_hour(logs_ops_stress))",
        "",
        "# restore",
        "RECALL_FLOOR.clear(); RECALL_FLOOR.update(orig_RF)",
        "FP_BUDGET_PER_H.clear(); FP_BUDGET_PER_H.update(orig_FP)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- KPIs (VAL-only) + PR curves + Confusion Matrix; save to ARTS ----",
        "def pager_load(logs, ts_all):",
        "    pages = [p for p in logs[\"proposals\"] if p[\"action\"] in [\"REQUEST_CONSULT\",\"REQUEST_BED\"]]",
        "    if not pages: return {\"pages\":0,\"per_hour\":0.0}",
        "    hours = max(1.0, (ts_all.max() - ts_all.min())/60.0)",
        "    return {\"pages\":len(pages), \"per_hour\": round(len(pages)/hours, 3)}",
        "",
        "def simulate_capacity_updates_list(total_minutes):",
        "    t=0; out=[]",
        "    while t<total_minutes:",
        "        out.append(t); t+=int(np.random.uniform(10,20))",
        "    return out",
        "",
        "def kpis_from_logs(logs, ts_all):",
        "    props = logs[\"proposals\"]; appr = logs[\"approvals\"]",
        "    approve_rate = len(appr)/max(1,len(props))",
        "    page_props = [p for p in props if p[\"action\"] in [\"REQUEST_CONSULT\",\"REQUEST_BED\"]]",
        "    page_appr  = [a for a in appr  if a[\"action\"] in [\"REQUEST_CONSULT\",\"REQUEST_BED\"]]",
        "    fpr = (len(page_props)-len(page_appr))/max(1,len(page_props))",
        "    caps = simulate_capacity_updates_list(SIM_MINUTES)",
        "    saved = []",
        "    for p in props:",
        "        last_update = max([c for c in caps if c <= p[\"ts\"]], default=None)",
        "        saved.append(minutes_saved_for(p[\"action\"], p[\"ts\"], last_update))",
        "    med_saved = float(np.median(saved)) if saved else 0.0",
        "    return {\"proposals\":len(props),\"approvals\":len(appr),\"approve_rate\":round(approve_rate,3),",
        "            \"false_page_rate\":round(fpr,3), \"median_minutes_saved\":round(med_saved,2)}",
        "",
        "kpi_report = {\"PR_cost\": kpis_from_logs(logs_cost, tv_all),",
        "              \"OPS\":      kpis_from_logs(logs_ops,  tv_all)}",
        "(ARTS/'kpi_report.json').write_text(json.dumps(kpi_report, indent=2))",
        "print(\"KPI report:\", kpi_report)",
        "print(\"Pager load (PR):\", pager_load(logs_cost, tv_all))",
        "print(\"Pager load (OPS):\", pager_load(logs_ops,  tv_all))",
        "",
        "from sklearn.metrics import precision_recall_curve",
        "critical = ['ORDER_CT','PERFORM_FAST','ORDER_ECG']",
        "plt.figure()",
        "for name in critical:",
        "    ci = ACTIONS.index(name)",
        "    yb = (yv_all==ci).astype(int)",
        "    if yb.sum() == 0:",
        "        print(f\"[PR] Skipping {name}: zero positives in VAL\")",
        "        continue",
        "    prec, rec, thr = precision_recall_curve(yb, probs_val[:,ci])",
        "    plt.plot(rec, prec, label=name)",
        "plt.xlabel(\"Recall\"); plt.ylabel(\"Precision\"); plt.title(\"PR Curves (critical actions)\"); plt.legend()",
        "plt.tight_layout(); plt.savefig(ARTS/'pr_curves_val.png', dpi=160); plt.close()",
        "",
        "y_pred = probs_val.argmax(axis=1)",
        "cm = np.zeros((len(ACTIONS), len(ACTIONS)), dtype=int)",
        "for t,p in zip(yv_all, y_pred): cm[t,p]+=1",
        "plt.figure()",
        "plt.imshow(cm, interpolation=\"nearest\")",
        "plt.title(\"Val Confusion Matrix\"); plt.xlabel(\"Pred\"); plt.ylabel(\"True\")",
        "plt.xticks(range(len(ACTIONS)), ACTIONS, rotation=45, ha=\"right\"); plt.yticks(range(len(ACTIONS)), ACTIONS)",
        "plt.colorbar()",
        "plt.tight_layout(); plt.savefig(ARTS/'cm_val.png', dpi=160); plt.close()",
        "print(\"Saved plots:\", str(ARTS/'pr_curves_val.png'), str(ARTS/'cm_val.png'))"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "outputs": [],
      "source": [
        "# ---- Run summary + audit anchor + artifact list ----",
        "summary = {",
        "    \"preset\": PRESET,",
        "    \"target_daily_arrivals\": TARGET_DAILY_ARRIVALS,",
        "    \"lambda_scale\": float(LAMBDA_SCALE),",
        "    \"n_days\": N_DAYS,",
        "    \"top_k\": TOP_K,",
        "    \"use_focal\": USE_FOCAL,",
        "    \"gamma\": GAMMA,",
        "    \"recall_floor\": RECALL_FLOOR,",
        "    \"fp_budget_per_h\": FP_BUDGET_PER_H,",
        "    \"max_cap_age_min\": MAX_CAP_AGE_MIN,",
        "    \"capacity_decay\": CAPACITY_DECAY,",
        "    \"hard_cap_pages_per_2h\": HARD_CAP_PAGES_PER_2H",
        "}",
        "(ARTS/'run_summary.json').write_text(json.dumps(summary, indent=2))",
        "",
        "import hashlib, time",
        "def file_hash(path):",
        "    h=hashlib.sha256()",
        "    with open(path,\"rb\") as f:",
        "        for chunk in iter(lambda: f.read(8192), b\"\"):",
        "            h.update(chunk)",
        "    return h.hexdigest()",
        "",
        "asset_paths = sorted([str(p) for p in ARTS.glob(\"*.*\")])",
        "leaves = [file_hash(p) for p in asset_paths]",
        "layer = leaves[:]",
        "if not layer: root = hashlib.sha256(b\"\").hexdigest()",
        "else:",
        "    while len(layer)>1:",
        "        nxt = []",
        "        it = iter(layer)",
        "        for a in it:",
        "            b = next(it, a)",
        "            nxt.append(hashlib.sha256((a+b).encode()).hexdigest())",
        "        layer = nxt",
        "    root = layer[0]",
        "",
        "anchor = {\"root\":root,\"ts\":time.strftime(\"%Y-%m-%dT%H:%M:%SZ\")}",
        "with open(ARTS/\"audit_anchor.log\",\"a\") as f: f.write(json.dumps(anchor)+\"\\n\")",
        "print(\"Anchored:\", anchor)",
        "",
        "def list_artifacts(arts_dir):",
        "    rows=[]",
        "    for p in sorted(Path(arts_dir).glob(\"*\")):",
        "        if p.is_file():",
        "            rows.append((p.name, p.stat().st_size))",
        "    return rows",
        "",
        "files = list_artifacts(ARTS)",
        "print(\"Artifacts:\", files)"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.11",
      "mimetype": "text/x-python",
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "pygments_lexer": "ipython3",
      "nbconvert_exporter": "python",
      "file_extension": ".py"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}


{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# ED Ops Pipeline (Kaggle/Papermill) — Sim → Train (Focal) → Cal → Threshold (PR/OPS) → Replay (Top-K) → KPIs → Scenarios → Stress/Spot-Checks\n',
    '\n',
    '**Kaggle-ready**: kernelspec metadata, non-interactive plotting, env-aware paths, no widgets.\n',
    'Artifacts → **`/kaggle/working/artifacts/`** (or `./artifacts`).\n',
    '\n',
    '**What’s new in v2b**\n',
    '- 3 replays with explicit gate/policy reasons; **ownership** (BED after CONSULT), duplicate & stale capacity blocks.\n',
    '- Minutes-saved knobs: `MAX_CAP_AGE_MIN` (default 30) and `CAPACITY_DECAY ∈ {linear, convex2}`.\n',
    '- Pager burden: pages/hour histogram, optional **hard cap per 2-hour window**, hour-aware budgets hook saved to notes.\n',
    '- Safety: per-action precision/recall @ τ (OPS), τ sanity, misroutes table.\n',
    '- **Rule spot-check**: verifies that hand rules trigger the model’s action ≥80% of the time.\n',
    '- *